# Mount Google Drive

In [21]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
print("Google Drive (drive) смонтирован.")

Mounted at /content/drive
Google Drive (drive) смонтирован.


# Установка зависимостей

In [22]:

print("--- Устанавливаю зависимости... ---")
!pip install sqlalchemy psycopg2-binary easyocr opencv-python-headless --quiet
print("--- Зависимости установлены. ---")


--- Устанавливаю зависимости... ---
--- Зависимости установлены. ---


# Импорты и Секреты

In [23]:

import sys
import re
import os
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo
import math
import cv2
import easyocr
import logging
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, Column, Integer, Float, Date, TIMESTAMP, ForeignKey, Text, func
from sqlalchemy.orm import sessionmaker, relationship, declarative_base, Session
from sqlalchemy.schema import UniqueConstraint, Index
from sqlalchemy.dialects.postgresql import insert
from google.colab import userdata
from sqlalchemy.exc import IntegrityError
import gspread
from oauth2client.service_account import ServiceAccountCredentials

print("--- Импорты завершены. ---")

--- Импорты завершены. ---


# Логгер

In [24]:
# --- Настройка Логгирования ---
logger = logging.getLogger("DE_Pipeline_Autotest")
logger.setLevel(logging.INFO)
logger.propagate = False
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if (logger.hasHandlers()):
    logger.handlers.clear()
logger.addHandler(handler)
logger.info("Логгер настроен (уровень INFO).")



2026-02-16 18:56:29,259 - INFO - Логгер настроен (уровень INFO).


# Config

In [25]:
CONFIG = {
    # OCR
    "debug_viz": False, #при True в выводе будут видны обработанные картинки
    "root_folder": "/content/drive/MyDrive/OCR Retriever Project/Test Charts",
    "year_for_parsing": "2025",
    "crop": {
        "y2_ratio": 0.24,  # 24% по высоте
        "x2_ratio": 0.60,  # 60% по ширине
    },

    # Autotest
    "autotest_sheet_id": "198x3oQ9jPlRaWf2l3gzr9PsAGr68dhfPj2a7K4V9q5g",
    "required_columns": [
        "Test launch",
        "Ad Name",
        "Team",
        "% bench 1 этап",
        "Impressions",
        "Installs",
        "Clicks",
        "Ret 2d, %"
    ],

    "percent_columns": [
        "Ret 2d, %"
    ],

    "int_columns": [
        "Impressions",
        "Installs",
        "Clicks",
        "% bench 1 этап"
    ]
}



# Подключение к БД (engine/session/Base)

In [26]:
logger.info("Подключаюсь к БД...")
try:
    DATABASE_URL = userdata.get('DATABASE_URL')
    if not DATABASE_URL or not DATABASE_URL.startswith("postgresql://"):
        logger.error("❌ Секрет 'DATABASE_URL' не найден или имеет неверный формат.")
        sys.exit(1)

    engine = create_engine(DATABASE_URL, pool_pre_ping=True)
    SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
    Base = declarative_base()
    logger.info("✅ Успешно создано подключение к БД.")
except Exception as e:
    logger.error(f"❌ Ошибка подключения к БД: {e}")
    sys.exit(1)

2026-02-16 18:56:29,279 - INFO - Подключаюсь к БД...
2026-02-16 18:56:29,738 - INFO - ✅ Успешно создано подключение к БД.


# SQLAlchemy модели (таблицы)

In [27]:
# --- Описываем таблицы как Python-классы ---
logger.info("Определяю модели БД...")
class Project(Base):
    __tablename__ = 'projects'
    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(Text, unique=True, nullable=False)
    created_at = Column(TIMESTAMP, server_default=func.now())
    creatives = relationship("Creative", back_populates="project")

class Creative(Base):
    __tablename__ = 'creatives'
    video_id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(Text, nullable=False)
    project_id = Column(Integer, ForeignKey('projects.id', ondelete="CASCADE"), nullable=False)
    created_at = Column(TIMESTAMP, server_default=func.now())

    project = relationship("Project", back_populates="creatives")
    hook_hold_metrics = relationship("HookHoldMetrics", back_populates="creative")
    auto_test_metrics = relationship("AutoTestMetrics", back_populates="creative")

    __table_args__ = (
        UniqueConstraint('project_id', 'name', name='_project_creative_uc'),
    )

class HookHoldMetrics(Base):
    __tablename__ = 'hook_hold_metrics'
    id = Column(Integer, primary_key=True, autoincrement=True)
    video_id = Column(Integer, ForeignKey('creatives.video_id', ondelete="CASCADE"), nullable=False)
    hook = Column(Float)
    hold = Column(Float)
    date = Column(Date, nullable=False)
    created_at = Column(TIMESTAMP, server_default=func.now())

    creative = relationship("Creative", back_populates="hook_hold_metrics")

    __table_args__ = (
        UniqueConstraint('video_id', 'date', name='_video_date_uc'),
    )

class AutoTestMetrics(Base):
    __tablename__ = 'auto_test_metrics'

    id = Column(Integer, primary_key=True, autoincrement=True)
    video_id = Column(Integer, ForeignKey('creatives.video_id', ondelete="CASCADE"), nullable=False)
    date = Column(Date, nullable=False)
    team = Column(Text, nullable=False)

    bench = Column(Integer)          # может быть NULL из-за '-'
    impressions = Column(Integer)
    clicks = Column(Integer)
    installs = Column(Integer)

    retention = Column(Float)        # Ret 2d, % в формате 0..1

    created_at = Column(TIMESTAMP, server_default=func.now())

    creative = relationship("Creative", back_populates="auto_test_metrics")

    __table_args__ = (
        UniqueConstraint('video_id', 'date', 'team', name='_video_date_team_uc'),
    )


logger.info("Проверяю/создаю таблицы в БД...")
Base.metadata.create_all(bind=engine)
logger.info("✅ Таблицы в БД проверены/созданы.")

2026-02-16 18:56:29,752 - INFO - Определяю модели БД...
2026-02-16 18:56:29,765 - INFO - Проверяю/создаю таблицы в БД...
2026-02-16 18:56:30,349 - INFO - ✅ Таблицы в БД проверены/созданы.


# Утилиты для БД (Get-or-Create / Upsert)

In [28]:

logger.info("Определяю утилиты для БД...")

def get_or_create_project(session: Session, project_name: str) -> int:
    project_name = (project_name or "").strip()
    if not project_name:
        raise ValueError("project_name пустой")

    session.execute(
        insert(Project)
        .values(name=project_name)
        .on_conflict_do_nothing(index_elements=["name"])
    )

    row = session.query(Project.id).filter_by(name=project_name).one_or_none()
    if row is None:
        raise RuntimeError(f"Project не найден после insert/conflict: '{project_name}'")
    return row[0]


def get_or_create_creative(session: Session, project_id: int, creative_name: str) -> int:
    creative_name = (creative_name or "").strip()
    if not creative_name:
        raise ValueError("creative_name пустой")

    session.execute(
        insert(Creative)
        .values(project_id=project_id, name=creative_name)
        .on_conflict_do_nothing(index_elements=["project_id", "name"])
    )

    row = (
        session.query(Creative.video_id)
        .filter_by(project_id=project_id, name=creative_name)
        .one_or_none()
    )
    if row is None:
        raise RuntimeError(
            f"Creative не найден после insert/conflict: project_id={project_id}, name='{creative_name}'"
        )
    return row[0]


def upsert_hook_hold_metric(session: Session, video_id: int, metric_date, hook_value: float, hold_value: float):
    stmt = insert(HookHoldMetrics).values(
        video_id=video_id,
        date=metric_date,
        hook=hook_value,
        hold=hold_value
    )
    stmt = stmt.on_conflict_do_update(
        index_elements=['video_id', 'date'],
        set_={
            'hook': stmt.excluded.hook,
            'hold': stmt.excluded.hold
        }
    )
    session.execute(stmt)

def upsert_auto_test_metric(session: Session, video_id: int, record: dict):
    metric_date = record.get("Test launch")
    team = (record.get("Team") or "").strip()
    if not metric_date:
        raise ValueError("Test launch пустой/не распарсился")
    if not team:
        raise ValueError("Team пустой")

    values = {
        "video_id": video_id,
        "date": metric_date,
        "team": team,
        "bench": record.get("% bench 1 этап"),
        "impressions": record.get("Impressions"),
        "clicks": record.get("Clicks"),
        "installs": record.get("Installs"),
        "retention": record.get("Ret 2d, %"),  # уже 0..1 после нормализации
    }

    stmt = insert(AutoTestMetrics).values(**values)
    stmt = stmt.on_conflict_do_update(
        index_elements=["video_id", "date", "team"],
        set_={
            "bench": stmt.excluded.bench,
            "impressions": stmt.excluded.impressions,
            "clicks": stmt.excluded.clicks,
            "installs": stmt.excluded.installs,
            "retention": stmt.excluded.retention,
        },
    )
    session.execute(stmt)


logger.info("✅ Утилиты для БД определены.")

2026-02-16 18:56:30,366 - INFO - Определяю утилиты для БД...
2026-02-16 18:56:30,369 - INFO - ✅ Утилиты для БД определены.


# Настройка EasyOCR и функции OCR скрипта

In [29]:
# --- Настройка EasyOCR ---
try:
    reader = easyocr.Reader(['en'])
    logger.info("✅ EasyOCR Reader успешно инициализирован.")
except Exception as e:
    logger.error(f"❌ Не удалось инициализировать EasyOCR: {e}")
    sys.exit(1)


# --- Функции OCR ---
logger.info("Определяю функции OCR...")

def crop(image, crop_cfg):
  height, width = image.shape[:2]
  y2 = int(crop_cfg["y2_ratio"] * height)
  x2 = int(crop_cfg["x2_ratio"] * width)
  return image[:y2, :x2]

def vizualise(image, file_path):
    """
    Показывает изображение (обрезанное) прямо в ячейке Colab.
    """
    try:

        logger.info(f"Визуализация: {os.path.basename(file_path)}")
        img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 5))
        plt.imshow(img_rgb)
        plt.title(f"Изображение для OCR: {os.path.basename(file_path)}")
        plt.axis('off')
        plt.show()
    except Exception as e:
        logger.warning(f"Не удалось визуализировать изображение: {e}")


def ocr_easy(file_path: str, debug_viz: bool = False):
    try:
        img = cv2.imread(file_path)
        if img is None:
            logger.error(f"Не удалось прочитать файл {file_path}")
            return None, None
        croped_img = crop(img, CONFIG["crop"])
        if debug_viz:
          vizualise(croped_img, file_path)
        results = reader.readtext(croped_img)
        text = " ".join([res[1] for res in results])
        return text, results
    except Exception as e:
        logger.exception(f"Ошибка в ocr_easy для {file_path}")
        return None, None

logger.info("✅ Функции OCR определены.")



2026-02-16 18:56:33,191 - INFO - ✅ EasyOCR Reader успешно инициализирован.
2026-02-16 18:56:33,192 - INFO - Определяю функции OCR...
2026-02-16 18:56:33,197 - INFO - ✅ Функции OCR определены.


# Главный OCR скрипт

In [30]:
# --- ГЛАВНЫЙ СКРИПТ ---
logger.info("Определяю главный скрипт...")

# --- Настройки ---
root_folder = CONFIG["root_folder"]

filename_pattern = re.compile(r"^([a-zA-Z0-9-]+)_([a-zA-Z0-9-]+)_(\d+x\d+)_([a-zA-Z]{2})\..*$")
YEAR_FOR_PARSING = CONFIG["year_for_parsing"]

def get_latest_date_from_db(session: Session) -> date:
    logger.info("Запрашиваю последнюю дату из БД (hook_hold_metrics)...")
    latest_date = session.query(func.max(HookHoldMetrics.date)).scalar()

    if latest_date:
        logger.info(f"✅ Самая поздняя дата в БД: {latest_date}")
        return latest_date
    else:
        logger.warning("Таблица hook_hold_metrics пуста. Устанавливаю дату на 2000-01-01.")
        return date(2000, 1, 1)

def main_ocr_script():
    tz_moscow = ZoneInfo("Europe/Moscow")
    today_str = datetime.now(tz_moscow).strftime("%Y-%m-%d %H:%M:%S")
    logger.info(f"\n{'-'*50}\n{today_str} - НАЧАЛО ОБРАБОТКИ OCR\n{'-'*50}")

    error_files = []
    new_records_saved = 0

    session = SessionLocal()
    try:
        if not os.path.exists(root_folder):
            logger.error(f"❌ КРИТИЧЕСКАЯ ОШИБКА: Путь не найден: {root_folder}")
            logger.error("Пожалуйста, примонтируйте Google Drive и проверьте путь.")
            return

        latest_date_in_db = get_latest_date_from_db(session)
        latest_month_start = latest_date_in_db.replace(day=1)

        for month_folder in sorted(os.listdir(root_folder)):
            month_path = os.path.join(root_folder, month_folder)

            if not (os.path.isdir(month_path) and re.match(r"^\d{2}\.\d{2}$", month_folder)):
                continue

            try:
                month_datetime = datetime.strptime("01." + month_folder, "%d.%m.%y").date()
            except ValueError:
                continue

            if month_datetime < latest_month_start:
                continue

            logger.info(f"\n=== Обрабатываю месяц {month_folder} ===")

            for day_folder in sorted(os.listdir(month_path)):
                day_path = os.path.join(month_path, day_folder)

                if not re.match(r"^\d{2}\.\d{2}$", day_folder):
                    continue

                try:
                    day_datetime = datetime.strptime(
                        f"{day_folder}.{YEAR_FOR_PARSING}",
                        "%d.%m.%Y"
                    ).date()
                except ValueError:
                    logger.warning(f"Неверный формат папки дня: {day_folder}. Пропуск.")
                    continue

                if day_datetime <= latest_date_in_db:
                    continue

                logger.info(f"\n>>> Обрабатываю дату {day_datetime} ({month_folder}/{day_folder})\n")

                day_saved = 0

                # --- ОБРАБОТКА ВСЕХ ФАЙЛОВ ДНЯ ---
                try:
                    for filename in sorted(os.listdir(day_path)):
                        match = filename_pattern.match(filename)
                        if not match:
                            continue

                        project_name = match.group(1)
                        creative_name = match.group(2)
                        img_path = os.path.join(day_path, filename)

                        logger.info(f"--- НОВЫЙ ФАЙЛ: {filename}")

                        try:
                            text, _ = ocr_easy(img_path, debug_viz=CONFIG["debug_viz"])
                            if not text:
                                logger.warning("Ошибка OCR: текст пустой. Пропуск.")
                                error_files.append(img_path)
                                continue

                            m = re.search(r"(\d+[.,]?\d*)\s*%\D+(\d+[.,]?\d*)\s*%", text)
                            if not m:
                                logger.warning(f"Ошибка: Не найден шаблон процентов: '{text}'. Пропуск.")
                                error_files.append(img_path)
                                continue

                            hook = float(m.group(1).replace(",", "."))
                            hold = float(m.group(2).replace(",", "."))

                            if not (0 <= hook <= 100 and 0 <= hold <= 100):
                              logger.warning(f"Ошибка OCR: проценты вне диапазона 0..100: hook={hook}, hold={hold}. Пропуск.")
                              error_files.append(img_path)
                              continue

                            project_id = get_or_create_project(session, project_name)
                            video_id = get_or_create_creative(session, project_id, creative_name)

                            upsert_hook_hold_metric(
                                session=session,
                                video_id=video_id,
                                metric_date=day_datetime,
                                hook_value=hook,
                                hold_value=hold
                            )

                            logger.info(
                                f"✅ Успех: {day_datetime}, Проект: {project_name}, "
                                f"Креатив: {creative_name}, {hook}%, {hold}%"
                            )

                            day_saved += 1
                            new_records_saved += 1

                        except Exception as file_error:
                            logger.error(f"Ошибка файла {img_path}: {file_error}. Пропуск.")
                            error_files.append(img_path)
                            continue

                    # --- КОММИТ ПО ДНЮ ---
                    if day_saved > 0:
                        logger.info(f"Коммичу день {day_datetime}: сохранено {day_saved} записей.")
                        session.commit()
                    else:
                        logger.info(f"День {day_datetime}: новых записей нет, commit не делаю.")

                except Exception as day_error:
                    # критический сбой дня (например, проблемы с папкой/IO)
                    logger.critical(f"Критическая ошибка дня {day_datetime}: {day_error}. Откатываю день.")
                    session.rollback()
                    continue

        # --- ИТОГОВЫЙ ОТЧЕТ (без commit, т.к. коммитим по дням) ---
        if error_files:
            logger.warning("!!!!!!!!!!!! Файлы с ошибками !!!!!!!!!!!!!!")
            for ef in error_files:
                logger.warning(f" - {ef}")

        logger.info(f"Готово. Всего успешно обработано записей: {new_records_saved}")

    except Exception as e:
        logger.critical(
            f"\n--- ❌ КРИТИЧЕСКАЯ ОШИБКА (СЕССИЯ): {e} ---\n"
            "Откатываю незакоммиченные изменения."
        )
        session.rollback()

    finally:
        session.close()
        logger.info("\n" + "=" * 50 + "\n")
        logger.info("--- СЕССИЯ OCR ЗАВЕРШЕНА ---")


# --- ЗАПУСК ГЛАВНОГО СКРИПТА ---
logger.info("✅ Все функции определены. Запускаю главный скрипт...")
main_ocr_script()

2026-02-16 18:56:33,247 - INFO - Определяю главный скрипт...
2026-02-16 18:56:33,253 - INFO - ✅ Все функции определены. Запускаю главный скрипт...
2026-02-16 18:56:33,254 - INFO - 
--------------------------------------------------
2026-02-16 21:56:33 - НАЧАЛО ОБРАБОТКИ OCR
--------------------------------------------------
2026-02-16 18:56:33,257 - INFO - Запрашиваю последнюю дату из БД (hook_hold_metrics)...
2026-02-16 18:56:33,330 - WARNING - Таблица hook_hold_metrics пуста. Устанавливаю дату на 2000-01-01.
2026-02-16 18:56:33,334 - INFO - 
=== Обрабатываю месяц 09.25 ===
2026-02-16 18:56:33,337 - INFO - 
>>> Обрабатываю дату 2025-09-30 (09.25/30.09)

2026-02-16 18:56:33,340 - INFO - --- НОВЫЙ ФАЙЛ: TR_Video-10_1080x1350_EN.mp4.png
2026-02-16 18:56:38,164 - INFO - ✅ Успех: 2025-09-30, Проект: TR, Креатив: Video-10, 53.0%, 33.2%
2026-02-16 18:56:38,165 - INFO - --- НОВЫЙ ФАЙЛ: TR_Video-11_1080x1350_EN.mp4.png
2026-02-16 18:56:41,673 - INFO - ✅ Успех: 2025-09-30, Проект: TR, Креатив: 

# Google Sheets: авторизация (gspread)

In [31]:
# --- Настройка Google Sheets (gspread) ---
logger.info("Подключаюсь к Google Sheets...")
try:
    creds_json = userdata.get('GOOGLE_CREDS_JSON')

    with open("creds.json", "w") as f:
        f.write(creds_json)

    scope = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
    ]
    creds = ServiceAccountCredentials.from_json_keyfile_name("creds.json", scope)
    gc = gspread.authorize(creds)
    logger.info("✅ Успешно подключился к Google Sheets.")
except Exception as e:
    logger.error(f"❌ Ошибка подключения к Google Sheets: {e}")
    logger.error("Проверь, что ты добавил секрет 'GOOGLE_CREDS_JSON' и 'поделился' таблицей.")
    sys.exit(1)

# ID "закрытой" таблицы (ОТКУДА ЧИТАТЬ)
CLOSED_SHEET_ID = CONFIG["autotest_sheet_id"]

2026-02-16 18:57:25,433 - INFO - Подключаюсь к Google Sheets...
2026-02-16 18:57:25,889 - INFO - ✅ Успешно подключился к Google Sheets.


# Sheets helpers: даты и валидация чисел

In [32]:

logger.info("Определяю вспомогательные функции...")

def _parse_date(date_str):
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except ValueError:
        return None


def check_numeric_columns(data, int_columns, percent_columns=None, max_warn=20):
    """
    Приводит данные к типам:
      - int_columns -> int|None
      - percent_columns -> float|None (в долях 0..1)

    Особенности:
      - запятая как десятичный разделитель поддерживается
      - пусто -> None
      - '-' допустим только для '% bench 1 этап' (ставим None), иначе строка считается невалидной
      - не мутирует исходные record (делает копию)
      - ограничивает количество warning-логов (max_warn)
    """
    if percent_columns is None:
        percent_columns = []

    logger.info(f"🔍 Валидация и нормализация: {len(data)} строк...")
    valid = []
    warn_count = 0

    def _to_str(v):
        return str(v).strip().replace(",", ".")

    for i, record in enumerate(data, start=1):
        clean = dict(record)
        try:
            # --- INT columns ---
            for col in int_columns:
                val = _to_str(clean.get(col, ""))
                if val == "-":
                    if col == "% bench 1 этап":
                        clean[col] = None
                        continue
                    raise ValueError(f"символ '-' недопустим в '{col}'")

                if val == "" or val.lower() == "none":
                    clean[col] = None
                    continue

                clean[col] = int(float(val))

            # --- PERCENT columns (as float 0..1) ---
            for col in percent_columns:
                val = _to_str(clean.get(col, ""))
                if val == "-" or val == "" or val.lower() == "none":
                    clean[col] = None
                    continue

                value = float(val)
                clean[col] = value
            valid.append(clean)

        except Exception as e:
            if warn_count < max_warn:
                logger.warning(f"⚠️ Строка {i} пропущена: {e}. Данные: {record}")
            warn_count += 1

    if warn_count > max_warn:
        logger.warning(f"⚠️ Пропущено строк с ошибками: {warn_count} (показаны первые {max_warn})")

    logger.info(f"✅ Валидация завершена: {len(valid)} строк валидны.")
    return valid



def fetch_autotest_data(gc, sheet_id, date_range_str, required_columns, int_columns, percent_columns):
    logger.info("--- Начинаю загрузку из Google Sheets ---")
    start_date = None
    end_date = None

    if ":" in date_range_str:
        parts = date_range_str.split(":")
        start_date = _parse_date(parts[0].strip())
        end_date = _parse_date(parts[1].strip())

    if start_date:
        logger.info(f"➡️ Загрузка начиная с {start_date}")
    else:
        logger.info("➡️ Загрузка всех данных (диапазон не указан).")

    try:
        # 1. Открываем Таблицу (Файл)
        sh = gc.open_by_key(sheet_id)
        # 2. Автоматически берем ПЕРВЫЙ лист (вкладку)
        ws = sh.sheet1
        # 3. Получаем его реальное имя
        project_name = ws.title
        logger.info(f"✅ Успешно открыл лист: '{project_name}'")
        all_data = ws.get_all_values()
        if not all_data:
            logger.warning("❌ Исходная таблица пуста.")
            return [], None # <-- Возвращаем None
        header, rows = all_data[0], all_data[1:]

        REQUIRED_COLUMNS = required_columns

        col_idx = {}
        for c in REQUIRED_COLUMNS:
            try:
                col_idx[c] = header.index(c)
            except ValueError:
                logger.error(f"❌ Ошибка: В исходной таблице не найден столбец '{c}'")
                return [], None # <-- Возвращаем None

    except Exception as e:
        logger.exception("❌ Ошибка чтения исходной таблицы")
        return [], None # <-- Возвращаем None

    filtered = []
    short_rows = 0
    max_idx = max(col_idx.values())
    for row in rows:
        if len(row) <= max_idx:
          short_rows += 1
          continue


        d = _parse_date(row[col_idx["Test launch"]])
        if not d: continue
        if start_date and d < start_date: continue
        if end_date and d > end_date: continue

        record = {c: row[idx] for c, idx in col_idx.items()}
        record["Test launch"] = d
        filtered.append(record)

    if short_rows:
      logger.warning(f"⚠️ Пропущено строк с недостаточным числом колонок: {short_rows}")
    logger.info(f"✅ Найдено {len(filtered)} строк из GSheets после фильтрации по дате.")

    filtered_data = check_numeric_columns(
        filtered,
        int_columns=int_columns,
        percent_columns=percent_columns
    )


    return filtered_data, project_name

logger.info("✅ Вспомогательные функции определены.")


2026-02-16 18:57:25,914 - INFO - Определяю вспомогательные функции...
2026-02-16 18:57:25,920 - INFO - ✅ Вспомогательные функции определены.


# Главный скрипт Sheets: настройки и запуск

In [33]:

logger.info("Определяю главный скрипт...")

# --- НАСТРОЙКИ ---
def get_latest_date_from_autotest_db(session: Session) -> date:
    """Запрашивает MAX(date) из таблицы auto_test_metrics."""
    logger.info("Запрашиваю последнюю дату из БД (auto_test_metrics)...")
    latest_date = session.query(func.max(AutoTestMetrics.date)).scalar()

    if latest_date:
        logger.info(f"✅ Самая поздняя дата в БД: {latest_date}")
        return latest_date
    else:
        logger.warning("Таблица auto_test_metrics пуста. Устанавливаю дату на начало 2000 года.")
        return date(2000, 1, 1)

def main_autotest_script(date_range_str=None):
#Можем передавать в функцию произвольно выбранный диапазон дат в формате %Y-%m-%d:%Y-%m-%d
    logger.info(f"\n{'-'*50}\nНАЧАЛО ОБРАБОТКИ AUTOTEST\n{'-'*50}")
    new_records_saved = 0
    session = SessionLocal()

    try:
      if date_range_str:
        # вручную поданный диапазон
        logger.info(f"📌 Использую РУЧНО заданный диапазон: {date_range_str}")
      else:
        latest_date_in_db = get_latest_date_from_autotest_db(session)
        start = latest_date_in_db + timedelta(days=1)
        date_range_str = f"{start.strftime('%Y-%m-%d')} :"

      # --- Шаг 2: Читаем НОВЫЕ данные ---
      data_to_upload, project_name_for_db = fetch_autotest_data(
          gc,
          CLOSED_SHEET_ID,
          date_range_str,
          required_columns=CONFIG["required_columns"],
          int_columns=CONFIG["int_columns"],
          percent_columns=CONFIG["percent_columns"]
      )

      if project_name_for_db is None:
          logger.error("❌ Ошибка чтения/схемы Google Sheet (пусто / не найдены колонки / ошибка доступа).")
          return

      if not data_to_upload:
          logger.info("ℹ️ Нет новых данных для загрузки из Google Sheets.")
          return

      else:
          logger.info(f"Получено {len(data_to_upload)} записей из листа '{project_name_for_db}'. Начинаю сохранение...")

          # --- Шаг 3: Получаем ID проекта ---
          project_id = get_or_create_project(session, project_name_for_db)

          # --- Шаг 4: Цикл по записям и UPSERT ---
          for record in data_to_upload:
              try:
                  creative_name = (record.get("Ad Name") or "").strip()
                  if not creative_name:
                      logger.warning(f"Пропущена строка: 'Ad Name' пустое. Данные: {record}")
                      continue

                  # 4.1. Get-or-Create Creative
                  video_id = get_or_create_creative(session, project_id, creative_name)

                  # 4.2. UPSERT Метрики
                  upsert_auto_test_metric(session, video_id, record)

                  new_records_saved += 1

              except Exception as e:
                  logger.error(f"❌ Ошибка при обработке строки: {e}. Данные: {record}")
                  continue

          # --- Шаг 5: Коммит транзакции ---
          if new_records_saved > 0:
              logger.info(f"Всего сохранено или обновлено {new_records_saved} записей. Коммичу изменения...")
              session.commit()
              logger.info("--- ✅ УСПЕХ! Новые данные успешно СОХРАНЕНЫ в БД. ---")
          else:
              logger.info("Новых данных для сохранения не найдено (возможно, все были с ошибками).")

    except Exception as e:
        logger.critical(f"\n--- ❌ КРИТИЧЕСКАЯ ОШИБКА (СЕССИЯ): {e} ---\nОткатываю транзакцию.")
        session.rollback()
    finally:
        session.close()
        logger.info("\n" + "="*50 + "\n")
        logger.info("--- СЕССИЯ AUTOTEST ЗАВЕРШЕНА ---")

# --- ЗАПУСК ГЛАВНОГО СКРИПТА ---
logger.info("✅ Все функции определены. Запускаю главный скрипт...")

# Вызываем главную функцию
main_autotest_script()

2026-02-16 18:57:25,936 - INFO - Определяю главный скрипт...
2026-02-16 18:57:25,939 - INFO - ✅ Все функции определены. Запускаю главный скрипт...
2026-02-16 18:57:25,940 - INFO - 
--------------------------------------------------
НАЧАЛО ОБРАБОТКИ AUTOTEST
--------------------------------------------------
2026-02-16 18:57:25,942 - INFO - Запрашиваю последнюю дату из БД (auto_test_metrics)...
2026-02-16 18:57:26,007 - WARNING - Таблица auto_test_metrics пуста. Устанавливаю дату на начало 2000 года.
2026-02-16 18:57:26,008 - INFO - --- Начинаю загрузку из Google Sheets ---
2026-02-16 18:57:26,009 - INFO - ➡️ Загрузка начиная с 2000-01-02
2026-02-16 18:57:26,790 - INFO - ✅ Успешно открыл лист: 'TR'
2026-02-16 18:57:27,011 - INFO - ✅ Найдено 15 строк из GSheets после фильтрации по дате.
2026-02-16 18:57:27,012 - INFO - 🔍 Валидация и нормализация: 15 строк...
2026-02-16 18:57:27,014 - INFO - ✅ Валидация завершена: 15 строк валидны.
2026-02-16 18:57:27,015 - INFO - Получено 15 записей из л